In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [3]:
fecha_corte = pd.to_datetime("2026-05-31")

# Rango de análisis 
fecha_inicio_12m = fecha_corte - pd.DateOffset(months=12)
fecha_inicio_24m = fecha_corte - pd.DateOffset(months=24)

In [4]:
# ===============================
# Datos de clientes y ventas
# ===============================

# Primera compra por ID_UNICO (cosecha)
df_cosecha = pd.read_sql(f"""
SELECT 
    ID_Cliente_Unico,
    MIN(Fecha) AS CosechaFecha
FROM Ventas_Comerssia.dbo.Ventas_Unificadas
WHERE ID_Cliente_Unico IS NOT NULL
GROUP BY ID_Cliente_Unico
HAVING MIN(Fecha) <= '{fecha_corte}'
""", engine)
df_cosecha["CosechaFecha"] = pd.to_datetime(df_cosecha["CosechaFecha"], errors="coerce")

# Ventas en ventana de 24 meses
query_ventas = f"""
SELECT 
    Cliente,
    ID_Cliente_Unico,
    Fecha,
    Venta_Neta AS Venta
FROM Ventas_Comerssia.dbo.Ventas_Unificadas
WHERE Fecha BETWEEN '{fecha_inicio_24m:%Y-%m-%d}' AND '{fecha_corte:%Y-%m-%d}'
"""

df_ventas = pd.read_sql(query_ventas, engine)
df_ventas["Fecha"] = pd.to_datetime(df_ventas["Fecha"], errors="coerce")

In [5]:
# ===============================
# Validacion de cosecha por ID
# ===============================
ids_totales = df_cosecha["ID_Cliente_Unico"].nunique()
ids_sin_cosecha = df_cosecha["CosechaFecha"].isna().sum()

print("\n📌 Validacion Cosecha por ID_Cliente_Unico")
print(f"  IDs unicos totales: {ids_totales:,}")
print(f"  IDs sin CosechaFecha: {ids_sin_cosecha:,} ({ids_sin_cosecha / max(len(df_cosecha), 1) * 100:.2f}%)")

if ids_totales > 0:
    print(f"  Primera cosecha registrada: {df_cosecha['CosechaFecha'].min()}")
    print(f"  Ultima cosecha registrada:  {df_cosecha['CosechaFecha'].max()}")


📌 Validacion Cosecha por ID_Cliente_Unico
  IDs unicos totales: 282,715
  IDs sin CosechaFecha: 0 (0.00%)
  Primera cosecha registrada: 2019-09-01 00:00:00
  Ultima cosecha registrada:  2026-05-31 00:00:00


In [6]:
# ===============================
# 3) Métricas por ID_UNICO
# ===============================

# Última compra
ultima_compra = (
    df_ventas.groupby("ID_Cliente_Unico")["Fecha"]
    .max()
    .reset_index()
    .rename(columns={"Fecha": "UltimaCompra"})
)

# Ventas últimos 12 meses (para Segmento Actual)
ventas_12m = (
    df_ventas[df_ventas["Fecha"] >= fecha_inicio_12m]
    .groupby("ID_Cliente_Unico")["Venta"]
    .sum()
    .reset_index()
    .rename(columns={"Venta": "Venta12M"})
)

# Ventas 24 meses (para Segmento24M)
ventas_24m = (
    df_ventas.groupby("ID_Cliente_Unico")["Venta"]
    .sum()
    .reset_index()
    .rename(columns={"Venta": "Venta24M"})
)

# ===============================
# 4) Merge: base cosecha + métricas por ID_UNICO
# ===============================
clientes = df_cosecha.copy()
clientes = clientes.merge(ultima_compra, on="ID_Cliente_Unico", how="left")
clientes = clientes.merge(ventas_12m, on="ID_Cliente_Unico", how="left")
clientes = clientes.merge(ventas_24m, on="ID_Cliente_Unico", how="left")

In [7]:
# ===============================
# 5) Función exacta para asignar segmento por monto 
# ===============================
def segmento_por_valor(valor):
    if pd.isna(valor) or valor == 0:
        return "Sin Segmento"
    elif valor > 1_400_000:
        return "Diamante"
    elif valor >= 700_000:
        return "Oro"
    elif valor >= 300_000:
        return "Plata"
    else:
        return "Bronce"

clientes["Segmento12M"] = clientes["Venta12M"].apply(segmento_por_valor)
clientes["Segmento24M"] = clientes["Venta24M"].apply(segmento_por_valor)

# ===============================
# 6) Recencia en días 
# ===============================
clientes["RecenciaDias"] = (fecha_corte - clientes["UltimaCompra"]).dt.days

def calcular_recencia(row):
    if pd.isna(row["UltimaCompra"]):
        return None

    if pd.notna(row["CosechaFecha"]):
        diff_cosecha = (fecha_corte - row["CosechaFecha"]).days
        if diff_cosecha <= 90:
            return "Nuevo"

    d = row["RecenciaDias"]

    if pd.isna(d):
        return None
    if d <= 120:
        return "Muy Activo"
    elif d <= 300:
        return "Activo"
    elif d <= 365:
        return "Por Inactivar"
    elif d <= 395:
        return "Churn"
    else:
        return "Inactivo"

clientes["Recencia"] = clientes.apply(calcular_recencia, axis=1)

# ===============================
# 7) Definir Segmento final 
# ===============================
def asignar_segmento_final(row):
    rec = row["Recencia"]
    if rec in ["Nuevo", "Muy Activo", "Activo", "Por Inactivar"]:
        return row["Segmento12M"]
    if rec in ["Churn", "Inactivo"]:
        return row["Segmento24M"]
    return "Sin Segmento"

clientes["Segmento"] = clientes.apply(asignar_segmento_final, axis=1)

# ===============================
# 8) Regla final: si Sin Segmento -> Recencia = None
# ===============================
clientes.loc[clientes["Segmento"] == "Sin Segmento", "Recencia"] = None

In [8]:
# ===============================
# 9) Resultado final: un registro por Cliente
#    con los datos de segmento de su ID_UNICO
# ===============================
df_master = pd.read_sql("""
SELECT Cliente, ID_Cliente_Unico
FROM Ventas_Comerssia.dbo.MASTER_ID
""", engine)

resultado_final = df_master.merge(
    clientes[[
        "ID_Cliente_Unico", "CosechaFecha", "UltimaCompra",
        "Venta12M", "Venta24M", "Recencia", "Segmento"
    ]],
    on="ID_Cliente_Unico",
    how="left"
)

In [9]:
# ===============================
# 10) Preview resultado final
# ===============================
print(f"Total filas (clientes): {len(resultado_final):,}")
print(f"IDs unicos: {resultado_final['ID_Cliente_Unico'].nunique():,}")
print(resultado_final.head(50))

Total filas (clientes): 408,974
IDs unicos: 389,642
         Cliente  ID_Cliente_Unico CosechaFecha UltimaCompra   Venta12M  \
0    C1020404373            100000   2026-04-29   2026-04-29  389915.96   
1    C1021633003            100001          NaT          NaT        NaN   
2    C1022034588            100002   2026-05-05   2026-05-05  237310.92   
3    C1013602736            100003          NaT          NaT        NaN   
4    C1020834450            100004   2026-05-01   2026-05-01  202689.07   
5    C1017132419            100005   2026-05-02   2026-05-02   79831.93   
6    C1022373856            100006   2026-05-05   2026-05-05  450168.06   
7    C1010184707            100007   2026-05-03   2026-05-03  109243.68   
8    C1020842545            100008   2026-05-03   2026-05-03  201680.67   
9    C1020836455            100009          NaT          NaT        NaN   
10   C1026295819            100010   2026-04-30   2026-04-30  252100.84   
11             C            100011   2021-01-09 

In [10]:
exportar = resultado_final[resultado_final["CosechaFecha"].notna()]
exportar.to_excel("segmentacion_ID_UNICO.xlsx", index=False)

In [11]:
exportar.to_sql("Segmentacion_ID_UNICO", engine, if_exists="replace", index=False)

177

In [12]:
# resultado.to_sql("Segmentacion_ID_UNICO", engine, if_exists="replace", index=False)